# Advanced EDA and Feature Selection

This notebook reviews the customer-level feature table with business-oriented visuals, then selects a simple first-baseline feature set for clustering. It does not run clustering, fit models, or create association rules.

## Imports and Data Loading

Use the interpretable feature table for EDA visuals and the preprocessed model table for feature selection.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
customer_features = pd.read_csv("../data/processed/customer_features.csv")
customer_features_model = pd.read_csv("../data/processed/customer_features_model.csv")

print(f"customer_features shape: {customer_features.shape}")
print(f"customer_features_model shape: {customer_features_model.shape}")

In [ ]:
customer_features.head()

## Customer Value

Review total customer value before looking at customer profiles. Large differences in total lifetime spend can strongly influence segmentation.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(customer_features["total_lifetime_spend"], bins=40, color="steelblue")
plt.title("Distribution of Total Lifetime Spend")
plt.xlabel("Total lifetime spend")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

In [ ]:
spend_quantiles = customer_features["total_lifetime_spend"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_frame(name="total_lifetime_spend")
spend_quantiles

In [ ]:
plt.figure(figsize=(10, 3))
sns.boxplot(data=customer_features, x="total_lifetime_spend", color="steelblue")
plt.title("Total Lifetime Spend Outliers")
plt.xlabel("Total lifetime spend")
plt.tight_layout()
plt.show()

Total lifetime spend is visibly uneven, with a smaller group of high-value customers far above the typical range. This suggests that customer value should be included in the baseline, but the scaled model table is important so high spend does not dominate by raw magnitude.

## Demographics and Lifecycle

Inspect age and tenure to see whether lifecycle differences may contribute to customer segments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customer_features["age"], bins=30, color="steelblue", ax=axes[0])
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Number of customers")

sns.histplot(customer_features["customer_tenure"], bins=30, color="seagreen", ax=axes[1])
axes[1].set_title("Customer Tenure Distribution")
axes[1].set_xlabel("Customer tenure")
axes[1].set_ylabel("Number of customers")

plt.tight_layout()
plt.show()

In [ ]:
plot_sample = customer_features.sample(n=min(5000, len(customer_features)), random_state=42)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_sample,
    x="age",
    y="total_lifetime_spend",
    hue="customer_tenure",
    palette="viridis",
    alpha=0.45,
    s=25,
    edgecolor=None,
)
plt.title("Age vs Total Lifetime Spend")
plt.xlabel("Age")
plt.ylabel("Total lifetime spend")
plt.legend(title="Tenure", loc="best")
plt.tight_layout()
plt.show()

Age and tenure show enough spread to be useful customer context. The scatter does not suggest a single simple age-spend pattern, so lifecycle features may help distinguish segments when combined with spending and basket behaviour.

## Loyalty and Promotions

Compare loyalty-card status and promotion sensitivity against customer value.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=customer_features, x="has_loyalty_card", y="total_lifetime_spend", color="steelblue")
plt.title("Total Lifetime Spend by Loyalty Card Status")
plt.xlabel("Has loyalty card")
plt.ylabel("Total lifetime spend")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customer_features["percentage_of_products_bought_promotion"], bins=35, color="steelblue", ax=axes[0])
axes[0].set_title("Promotion Purchase Percentage Distribution")
axes[0].set_xlabel("Promotion purchase percentage")
axes[0].set_ylabel("Number of customers")

sns.scatterplot(
    data=plot_sample,
    x="percentage_of_products_bought_promotion",
    y="total_lifetime_spend",
    alpha=0.4,
    s=25,
    color="seagreen",
    edgecolor=None,
    ax=axes[1],
)
axes[1].set_title("Promotion Percentage vs Total Lifetime Spend")
axes[1].set_xlabel("Promotion purchase percentage")
axes[1].set_ylabel("Total lifetime spend")

plt.tight_layout()
plt.show()

Loyalty-card status and promotion behaviour both look useful for separating customer types. Promotion values should remain in the selected baseline because they may identify promotion-sensitive customers, while loyalty status adds a simple relationship signal.

## Spending Preferences

Compare category-level spending mix. Category shares are especially useful because they describe preference rather than only customer size.

In [ ]:
share_columns = [column for column in customer_features.columns if column.startswith("share_")]
category_labels = [column.replace("share_", "") for column in share_columns]

avg_spend_by_category = (
    customer_features[share_columns]
    .multiply(customer_features["total_lifetime_spend"], axis=0)
    .mean()
    .rename(index=dict(zip(share_columns, category_labels)))
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_spend_by_category.values, y=avg_spend_by_category.index, color="steelblue")
plt.title("Average Estimated Spend by Category")
plt.xlabel("Average estimated spend")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
avg_share_by_category = (
    customer_features[share_columns]
    .mean()
    .rename(index=dict(zip(share_columns, category_labels)))
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_share_by_category.values, y=avg_share_by_category.index, color="seagreen")
plt.title("Average Spend Share by Category")
plt.xlabel("Average share of total spend")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
share_corr = customer_features[share_columns].corr()
share_corr.index = category_labels
share_corr.columns = category_labels
sns.heatmap(share_corr, cmap="vlag", center=0, linewidths=0.5)
plt.title("Spending Category Share Relationships")
plt.tight_layout()
plt.show()

Spending categories provide preference signals that are different from total spend. The share features should be kept for the first baseline because they can help distinguish grocery-heavy, electronics-heavy, hygiene-heavy, or entertainment-oriented profiles.

## Basket Behaviour

Basket features add transaction-level behaviour to the customer-level view.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customer_features["basket_count"], bins=30, color="steelblue", ax=axes[0])
axes[0].set_title("Basket Count Distribution")
axes[0].set_xlabel("Number of baskets")
axes[0].set_ylabel("Number of customers")

sns.histplot(customer_features["avg_basket_size"], bins=30, color="seagreen", ax=axes[1])
axes[1].set_title("Average Basket Size Distribution")
axes[1].set_xlabel("Average basket size")
axes[1].set_ylabel("Number of customers")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_sample,
    x="basket_count",
    y="total_lifetime_spend",
    alpha=0.45,
    s=25,
    color="steelblue",
    edgecolor=None,
)
plt.title("Basket Count vs Total Lifetime Spend")
plt.xlabel("Number of baskets")
plt.ylabel("Total lifetime spend")
plt.tight_layout()
plt.show()

Basket count and average basket size add behavioural context beyond lifetime spend. For the first baseline, basket count and average basket size are useful while more redundant basket totals can be left out.

## Correlation Overview

Review selected numerical candidates to avoid carrying too many highly related features into the first baseline.

In [ ]:
correlation_columns = [
    "total_lifetime_spend",
    "lifetime_total_distinct_products",
    "age",
    "customer_tenure",
    "percentage_of_products_bought_promotion",
    "distinct_stores_visited",
    "total_children_home",
    "basket_count",
    "avg_basket_size",
    "distinct_products_in_baskets",
]

plt.figure(figsize=(10, 8))
sns.heatmap(
    customer_features[correlation_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    linewidths=0.5,
)
plt.title("Correlation Overview for Candidate Numerical Features")
plt.tight_layout()
plt.show()

The basket totals and basket breadth measures are closely related to transaction frequency, so the first baseline can keep a smaller basket set. Tenure replaces raw first transaction year, and total children replaces the separate children indicators for a simpler starting point.

# Feature Selection for First Clustering Baseline

Start from `customer_features_model.csv`, then keep a compact, interpretable set of scaled and encoded features for the first clustering baseline.

In [ ]:
selected_feature_columns = [
    "customer_id",
    "number_complaints",
    "distinct_stores_visited",
    "lifetime_total_distinct_products",
    "percentage_of_products_bought_promotion",
    "typical_hour",
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "total_lifetime_spend",
    "share_groceries",
    "share_electronics",
    "share_vegetables",
    "share_nonalcohol_drinks",
    "share_alcohol_drinks",
    "share_meat",
    "share_fish",
    "share_hygiene",
    "share_videogames",
    "share_petfood",
    "basket_count",
    "avg_basket_size",
    "customer_gender_female",
    "degree_level_Bsc",
    "degree_level_Msc",
    "degree_level_Phd",
    "degree_level_Unknown",
]

selected_model_features = customer_features_model[selected_feature_columns].copy()
removed_feature_columns = [
    column for column in customer_features_model.columns
    if column not in selected_feature_columns
]

print(f"Selected columns: {len(selected_feature_columns):,}")
print(f"Removed columns: {len(removed_feature_columns):,}")

## Feature Selection Rationale

| Decision | Features | Reason |
|---|---|---|
| Keep | customer value, lifecycle, promotion, loyalty, store activity, spending shares, basket count/size, gender, degree level | Simple, interpretable baseline covering value, preferences, behaviour, and customer context. |
| Remove | `kids_home`, `teens_home`, `has_children` | Duplicated by `total_children_home` for a simpler baseline. |
| Remove | `year_first_transaction` | Duplicated by the more interpretable `customer_tenure`. |
| Remove | `latitude`, `longitude` | Potentially useful later, but risky for a first distance-based baseline because location can dominate clusters. |
| Remove | `max_basket_size`, `min_basket_size`, `total_items_bought_in_baskets`, `distinct_products_in_baskets`, `avg_distinct_products_per_basket` | Closely related to basket count or average basket size; kept a smaller basket behaviour set first. |
| Remove | `customer_gender_male` | Redundant with `customer_gender_female` in this binary gender encoding. |

In [ ]:
removed_features_table = pd.DataFrame({"removed_feature": removed_feature_columns})
removed_features_table

## Save Selected Features

Save the first-baseline selected feature table. This still does not run clustering.

In [ ]:
selected_model_features.to_csv("../data/processed/selected_model_features.csv", index=False)
print("Saved ../data/processed/selected_model_features.csv")

## Final Validation

Confirm the selected feature table is complete and keeps one row per customer.

In [ ]:
validation_summary = pd.DataFrame(
    {
        "check": [
            "row count",
            "duplicated customer_id count",
            "missing values count",
            "selected feature count including customer_id",
            "removed feature count",
        ],
        "value": [
            selected_model_features.shape[0],
            selected_model_features["customer_id"].duplicated().sum(),
            selected_model_features.isna().sum().sum(),
            selected_model_features.shape[1],
            len(removed_feature_columns),
        ],
    }
)

validation_summary

In [ ]:
selected_model_features.head()

The selected feature table is ready for a first clustering baseline. The next notebook should compare clustering options and evaluate segment quality without changing the raw data.